In [1]:
# =================================
# IMPORTS
# =================================

import os
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
from collections import Counter
import itertools

import spacy
from textblob import TextBlob

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

import networkx as nx
from wordcloud import WordCloud

# =================================
# LOAD NLP MODEL
# =================================

nlp = spacy.load("en_core_web_sm")

# =================================
# NYT DATA COLLECTION
# =================================

API_KEY = os.getenv("NYT_API_KEY")

url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

nyt_articles = []

for page in range(10):

    params = {
        "q": '"Trump" AND "Epstein"',
        "page": page,
        "api-key": API_KEY,
        "begin_date": "20200101",
        "end_date": "20260201"
    }

    r = requests.get(url, params=params, timeout=10)
    data = r.json()

    for doc in data["response"]["docs"]:

        nyt_articles.append({

            "source": "NYT",
            "headline": doc["headline"]["main"],
            "abstract": doc.get("abstract",""),
            "date": doc.get("pub_date","")

        })

nyt = pd.DataFrame(nyt_articles)

# =================================
# FOX NEWS SCRAPING
# =================================

fox_links = []

for page in range(1,15):

    search_url = f"https://www.foxnews.com/search-results/search?q=trump%20epstein&p={page}"

    r = requests.get(search_url, timeout=10)
    soup = BeautifulSoup(r.text,"html.parser")

    links = soup.find_all("a")

    for link in links:

        href = link.get("href")

        if href and "/politics/" in href:

            full = "https://www.foxnews.com"+href

            if full not in fox_links:
                fox_links.append(full)

fox_articles = []

for article_url in fox_links:

    try:

        r = requests.get(article_url, timeout=10)
        soup = BeautifulSoup(r.text,"html.parser")

        headline_tag = soup.find("h1")
        headline = headline_tag.get_text() if headline_tag else ""

        paragraph = soup.find("p")
        abstract = paragraph.get_text() if paragraph else ""

        fox_articles.append({

            "source":"FOX",
            "headline":headline,
            "abstract":abstract

        })

        time.sleep(2)

    except:
        pass

fox = pd.DataFrame(fox_articles)

# =================================
# BALANCE DATASET
# =================================

n = min(len(nyt),len(fox))

nyt = nyt.sample(n, random_state=42)
fox = fox.sample(n, random_state=42)

df = pd.concat([nyt,fox])

# =================================
# CLEANING
# =================================

df = df.dropna(subset=["headline","abstract"])

df["text"] = df["headline"] + " " + df["abstract"]

def clean(text):

    text = re.sub(r"\n"," ",text)
    text = re.sub(r"\s+"," ",text)

    return text.lower()

df["clean_text"] = df["text"].apply(clean)

df = df.drop_duplicates(subset=["clean_text"])

# =================================
# CORPUS CONSTRUCTION
# =================================

df["doc"] = list(nlp.pipe(df["clean_text"]))

# =================================
# NER ANALYSIS
# =================================

entities = []

for doc in df["doc"]:

    for ent in doc.ents:

        if ent.label_ == "PERSON":
            entities.append(ent.text)

top_entities = Counter(entities).most_common(10)

entities_df = pd.DataFrame(top_entities, columns=["entity","count"])

# =================================
# POS TAGGING
# =================================

pos_counts = []

for doc in df["doc"]:

    for token in doc:
        pos_counts.append(token.pos_)

print("Top POS:", Counter(pos_counts).most_common(10))

# =================================
# DEPENDENCY ANALYSIS
# =================================

roles_by_source = []

for index,row in df.iterrows():

    for token in row["doc"]:

        if token.text.lower() == "trump":

            roles_by_source.append({
                "source":row["source"],
                "role":token.dep_
            })

roles_df = pd.DataFrame(roles_by_source)

print(roles_df.groupby(["source","role"]).size())

# =================================
# SENTIMENT ANALYSIS
# =================================

def sentiment(text):
    return TextBlob(text).sentiment.polarity

df["sentiment"] = df["clean_text"].apply(sentiment)

print(df.groupby("source")["sentiment"].mean())

# =================================
# STATISTICAL TEST
# =================================

nyt_sent = df[df["source"]=="NYT"]["sentiment"]
fox_sent = df[df["source"]=="FOX"]["sentiment"]

t,p = ttest_ind(nyt_sent, fox_sent, equal_var=False)

print("T-statistic:", t)
print("P-value:", p)

# =================================
# VOCABULARY COMPARISON
# =================================

nyt_tokens = []
fox_tokens = []

for index,row in df.iterrows():

    for token in row["doc"]:

        if token.is_alpha and not token.is_stop:

            if row["source"] == "NYT":
                nyt_tokens.append(token.text)

            else:
                fox_tokens.append(token.text)

nyt_freq = Counter(nyt_tokens).most_common(15)
fox_freq = Counter(fox_tokens).most_common(15)

nyt_df = pd.DataFrame(nyt_freq, columns=["word","count"])
nyt_df["source"] = "NYT"

fox_df = pd.DataFrame(fox_freq, columns=["word","count"])
fox_df["source"] = "FOX"

vocab_df = pd.concat([nyt_df,fox_df])

# =================================
# KWIC CONTEXT
# =================================

def kwic(text, keyword="trump", window=5):

    words = text.split()
    contexts = []

    for i,w in enumerate(words):

        if w.lower() == keyword:

            left = " ".join(words[max(i-window,0):i])
            right = " ".join(words[i+1:i+window+1])

            contexts.append(left + " " + w + " " + right)

    return contexts

contexts = []

for text in df["clean_text"]:
    contexts += kwic(text)

print("Example contexts:", contexts[:10])

# =================================
# NARRATIVE FRAMES
# =================================

investigation_words = ["investigation","probe","court","case","evidence"]
defense_words = ["deny","defense","false","reject"]
association_words = ["friend","connection","associate","relationship"]

frames = []

for text in df["clean_text"]:

    words = text.split()

    frame = "neutral"

    if any(w in words for w in investigation_words):
        frame = "investigation"

    if any(w in words for w in defense_words):
        frame = "defense"

    if any(w in words for w in association_words):
        frame = "association"

    frames.append(frame)

df["frame"] = frames

print(df.groupby(["source","frame"]).size())

# =================================
# SENTIMENT VISUALIZATION
# =================================

plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="sentiment",
    hue="source",
    bins=20,
    kde=True
)

plt.title("Sentiment Distribution: NYT vs Fox News")
plt.show()

# =================================
# ENTITY VISUALIZATION
# =================================

plt.figure(figsize=(8,5))

sns.barplot(
    data=entities_df,
    x="count",
    y="entity"
)

plt.title("Most Mentioned People")
plt.show()

# =================================
# VOCABULARY VISUALIZATION
# =================================

plt.figure(figsize=(10,6))

sns.barplot(
    data=vocab_df,
    x="count",
    y="word",
    hue="source"
)

plt.title("Vocabulary Differences")
plt.show()

# =================================
# SYNTACTIC ROLE VISUALIZATION
# =================================

plt.figure(figsize=(8,5))

sns.countplot(
    data=roles_df,
    x="role",
    hue="source"
)

plt.title("Syntactic Roles of Trump")
plt.show()

# =================================
# ENTITY NETWORK
# =================================

pairs = []

for doc in df["doc"]:

    persons = [ent.text for ent in doc.ents if ent.label_=="PERSON"]

    for pair in itertools.combinations(persons,2):
        pairs.append(pair)

edges = Counter(pairs).most_common(20)

G = nx.Graph()

for (a,b),w in edges:
    G.add_edge(a,b,weight=w)

plt.figure(figsize=(10,8))
nx.draw(G, with_labels=True, node_size=2000)
plt.title("Network of People Mentioned Together")
plt.show()

# =================================
# FRAME HEATMAP
# =================================

frame_counts = df.groupby(["source","frame"]).size().unstack()

plt.figure(figsize=(8,5))

sns.heatmap(
    frame_counts,
    annot=True,
    cmap="coolwarm",
    fmt="g"
)

plt.title("Narrative Frames by News Source")
plt.show()

# =================================
# WORDCLOUD NYT
# =================================

nyt_text = " ".join(nyt_tokens)

wc = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(nyt_text)

plt.figure(figsize=(10,5))
plt.imshow(wc)
plt.axis("off")
plt.title("Vocabulary Emphasis in NYT")
plt.show()

# =================================
# WORDCLOUD FOX
# =================================

fox_text = " ".join(fox_tokens)

wc = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(fox_text)

plt.figure(figsize=(10,5))
plt.imshow(wc)
plt.axis("off")
plt.title("Vocabulary Emphasis in Fox News")
plt.show()

KeyError: 'response'